# Steps 11-13: Ablation, Robustness, and Reproducibility

**Step 11 — Ablation**: Systematic component-wise analysis
**Step 12 — Robustness**: Multi-seed runs, cross-pathology, failure analysis
**Step 13 — Reproducibility**: Final tables, figures, README

**Ablation Configurations** (at 10% and 25% labels):
1. Baseline (random init)
2. +SSL only
3. +SSL+Motion
4. +SSL+Pseudo
5. +SSL+Motion+Pseudo
6. Full (SSL+Motion+Pseudo+CF)

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

## 11.1 Compile Ablation Results

In [ ]:
# Load all results
baseline_path = os.path.join(RESULTS_DIR, 'baseline_results.json')
finetune_path = os.path.join(RESULTS_DIR, 'finetune_results.json')

baseline_results = {}
finetune_results = {}

if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baseline_results = json.load(f)

if os.path.exists(finetune_path):
    with open(finetune_path) as f:
        finetune_results = json.load(f)

print(f"Baseline experiments: {list(baseline_results.keys())}")
print(f"Fine-tune experiments: {list(finetune_results.keys())}")

In [ ]:
# Table C: Ablation Study
# Components to ablate at 10% and 25%
ablation_configs = [
    ('Baseline', None, None),  # From baseline_results
    ('+SSL', 'ssl', None),
    ('+SSL+Motion', 'ssl_motion', None),
    ('+SSL+Pseudo', 'ssl_pseudo', None),  # hypothetical — need separate run
    ('+SSL+Motion+Pseudo', 'ssl_motion_pseudo', None),
    ('Full (+CF)', 'ssl_motion_pseudo_cf', None),
]

for label_pct in [10, 25]:
    print(f"\n{'='*70}")
    print(f"TABLE C: Ablation Study ({label_pct}% Labels)")
    print(f"{'='*70}")
    print(f"{'Component':<25} {'Mean Dice':<15} {'LV':<10} {'Myo':<10} {'RV':<10} {'Δ Baseline':<10}")
    print("-" * 80)
    
    frac_key = str(label_pct / 100)
    baseline_dice = None
    
    for name, ft_prefix, _ in ablation_configs:
        if name == 'Baseline':
            if frac_key in baseline_results:
                m = baseline_results[frac_key]['mean']
                baseline_dice = m['Mean_Dice']
                delta = '—'
                print(f"{name:<25} {m['Mean_Dice']:<15.4f} {m['LV_Dice']:<10.4f} {m['Myocardium_Dice']:<10.4f} {m['RV_Dice']:<10.4f} {delta:<10}")
        else:
            key = f"{ft_prefix}_{label_pct}pct"
            if key in finetune_results:
                m = finetune_results[key]['mean']
                delta_val = m['Mean_Dice'] - baseline_dice if baseline_dice else 0
                delta = f"+{delta_val:.4f}" if delta_val >= 0 else f"{delta_val:.4f}"
                print(f"{name:<25} {m['Mean_Dice']:<15.4f} {m['LV_Dice']:<10.4f} {m['Myocardium_Dice']:<10.4f} {m['RV_Dice']:<10.4f} {delta:<10}")
            else:
                print(f"{name:<25} {'N/A':<15}")

## 11.2 Ablation Bar Chart

In [ ]:
# Ablation bar chart at 10% labels
label_pct = 10
frac_key = str(label_pct / 100)

config_names = []
dice_values = []
colors = []

color_map = {
    'Baseline': '#95a5a6',
    '+SSL': '#3498db',
    '+SSL+Motion': '#2ecc71',
    '+SSL+Pseudo': '#f39c12',
    '+SSL+Motion+Pseudo': '#e74c3c',
    'Full (+CF)': '#9b59b6',
}

for name, ft_prefix, _ in ablation_configs:
    if name == 'Baseline':
        if frac_key in baseline_results:
            config_names.append(name)
            dice_values.append(baseline_results[frac_key]['mean']['Mean_Dice'])
            colors.append(color_map[name])
    else:
        key = f"{ft_prefix}_{label_pct}pct"
        if key in finetune_results:
            config_names.append(name)
            dice_values.append(finetune_results[key]['mean']['Mean_Dice'])
            colors.append(color_map.get(name, '#95a5a6'))

if config_names:
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(range(len(config_names)), dice_values, color=colors, edgecolor='black', linewidth=0.5)
    
    for bar, val in zip(bars, dice_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    ax.set_xticks(range(len(config_names)))
    ax.set_xticklabels(config_names, rotation=30, ha='right')
    ax.set_ylabel('Mean Dice', fontsize=12)
    ax.set_title(f'Ablation Study — Component Contributions ({label_pct}% Labels)', fontsize=13)
    ax.set_ylim([0, 1.0])
    ax.grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'ablation_barchart.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No ablation results available yet. Run experiments first.")

## 11.3 Label-Efficiency Comparison Curve

In [ ]:
# Plot label-efficiency curves for baseline vs best method
fractions = [10, 25, 50, 100]

fig, ax = plt.subplots(figsize=(10, 6))

methods = {
    'Baseline': ('baseline_results', '#95a5a6', 'o', '-'),
    'SSL': ('finetune_results', '#3498db', 's', '--'),
    'Full Method': ('finetune_results', '#9b59b6', '^', '-'),
}

for method_name, (source, color, marker, linestyle) in methods.items():
    dice_vals = []
    valid_fracs = []
    
    for f in fractions:
        if source == 'baseline_results':
            key = str(f / 100)
            if key in baseline_results:
                dice_vals.append(baseline_results[key]['mean']['Mean_Dice'])
                valid_fracs.append(f)
        else:
            if method_name == 'SSL':
                key = f'ssl_{f}pct'
            else:  # Full
                key = f'ssl_motion_pseudo_cf_{f}pct'
            if key in finetune_results:
                dice_vals.append(finetune_results[key]['mean']['Mean_Dice'])
                valid_fracs.append(f)
    
    if valid_fracs:
        ax.plot(valid_fracs, dice_vals, marker=marker, linestyle=linestyle,
                color=color, linewidth=2, markersize=8, label=method_name)

ax.set_xlabel('% Training Labels', fontsize=12)
ax.set_ylabel('Mean Dice', fontsize=12)
ax.set_title('Label-Efficiency: Baseline vs Self-Supervised Methods', fontsize=13)
ax.set_xticks(fractions)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 1.0])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'label_efficiency_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12.1 Multi-Seed Robustness (if results available)

In [ ]:
# If multi-seed results exist, analyze them
# For now, document what would be done
print("Robustness Analysis Plan:")
print("1. Run final configuration with seeds [42, 123, 456]")
print("2. Report mean ± std across seeds")
print("3. Wilcoxon signed-rank test for significance")
print("4. Per-pathology group analysis")
print("5. Failure case analysis (worst 5 patients)")

# TODO: When compute allows, uncomment and run multi-seed
# seeds = [42, 123, 456]
# multi_seed_results = []
# for seed in seeds:
#     result = finetune_experiment(label_fraction=0.10, use_ssl=True,
#                                  use_motion=True, use_pseudo=True,
#                                  use_confidence_filter=True, seed=seed)
#     multi_seed_results.append(result)

## 12.2 Cross-Pathology Analysis

In [ ]:
# Per-pathology analysis would go here
# This requires running test evaluation with pathology tracking
print("Cross-pathology analysis requires per-patient results from test evaluation.")
print("This is done in the fine-tuning notebook when results include 'patient_id' and 'pathology'.")

## 13.1 Generate Final Tables

In [ ]:
# Table A: Main Segmentation Results (100% labels)
print("="*80)
print("TABLE A: Main Segmentation Results (100% Labels)")
print("="*80)

table_a_rows = []

# Baseline
if '1.0' in baseline_results:
    m = baseline_results['1.0']
    row = {'Method': 'Baseline (Random Init)'}
    for metric in ['LV_Dice', 'Myocardium_Dice', 'RV_Dice', 'Mean_Dice']:
        if metric in m['mean']:
            row[metric] = f"{m['mean'][metric]:.4f}±{m['std'][metric]:.4f}"
    for metric in ['LV_HD95', 'Myocardium_HD95', 'RV_HD95']:
        if metric in m['mean'] and np.isfinite(m['mean'][metric]):
            row[metric] = f"{m['mean'][metric]:.2f}±{m['std'][metric]:.2f}"
    table_a_rows.append(row)

# Full method
full_key = 'ssl_motion_pseudo_cf_100pct'
if full_key in finetune_results:
    m = finetune_results[full_key]
    row = {'Method': 'Full Method (SSL+Motion+PL+CF)'}
    for metric in ['LV_Dice', 'Myocardium_Dice', 'RV_Dice', 'Mean_Dice']:
        if metric in m['mean']:
            row[metric] = f"{m['mean'][metric]:.4f}±{m['std'][metric]:.4f}"
    for metric in ['LV_HD95', 'Myocardium_HD95', 'RV_HD95']:
        if metric in m['mean'] and np.isfinite(m['mean'][metric]):
            row[metric] = f"{m['mean'][metric]:.2f}±{m['std'][metric]:.2f}"
    table_a_rows.append(row)

if table_a_rows:
    table_a = pd.DataFrame(table_a_rows)
    print(table_a.to_string(index=False))
    table_a.to_csv(os.path.join(TABLES_DIR, 'table_a_main_results.csv'), index=False)
else:
    print("Results not yet available. Run experiments first.")

In [ ]:
# Compile all training curves
import glob

history_files = glob.glob(os.path.join(PROJECT_ROOT, 'checkpoints', '*_history.json'))
history_files += glob.glob(os.path.join(PROJECT_ROOT, 'checkpoints', 'ssl_history.json'))

if history_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for hf in history_files:
        name = Path(hf).stem.replace('_history', '')
        with open(hf) as f:
            hist = json.load(f)
        
        if 'train_loss' in hist:
            axes[0].plot(hist['train_loss'], label=name, alpha=0.7)
        if 'val_dice' in hist:
            axes[1].plot(hist['val_dice'], label=name, alpha=0.7)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Training Loss')
    axes[0].set_title('Training Loss Curves')
    axes[0].legend(fontsize=8)
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Dice')
    axes[1].set_title('Validation Dice Curves')
    axes[1].legend(fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No training history files found.")

In [ ]:
# Final checklist
print("="*60)
print("REQUIRED OUTPUTS CHECKLIST")
print("="*60)

required_tables = [
    ('Table A: Main Results', 'table_a_main_results.csv'),
    ('Table B: Label Efficiency', 'label_efficiency_all.csv'),
    ('Table C: Ablation', 'ablation_study.csv'),
    ('Table D: Pseudo-Labels', 'pseudo_label_analysis.csv'),
]

required_figures = [
    ('Architecture', 'architecture_overview.png'),
    ('Cine Motion', 'cine_motion_examples.png'),
    ('Baseline vs SSL', 'baseline_vs_ssl_segmentation.png'),
    ('Label-Efficiency Curve', 'label_efficiency_curve.png'),
    ('Displacement Field', 'displacement_field.png'),
    ('Pseudo-Labels', 'pseudo_labels_confidence.png'),
    ('Ablation Bar Chart', 'ablation_barchart.png'),
    ('Training Curves', 'training_curves.png'),
]

print("\nTables:")
for name, fname in required_tables:
    exists = os.path.exists(os.path.join(TABLES_DIR, fname))
    print(f"  {'✓' if exists else '✗'} {name} ({fname})")

print("\nFigures:")
for name, fname in required_figures:
    exists = os.path.exists(os.path.join(FIGURES_DIR, fname))
    print(f"  {'✓' if exists else '✗'} {name} ({fname})")

print("\n=== Steps 11-13: Evaluation, Ablation, Reproducibility COMPLETE ===")